# المهمة السادسة: نموذج أولي لتوقع الطلب الأسبوعي

يبني هذا الدفتر خط أساس تنبؤيًا لإجمالي الطلب الأسبوعي باستخدام `pandas` و`matplotlib` فقط. يُحجز أحدث 20% من الأسابيع للتحقق الزمني، وتُدرّب صيغة الاتجاه الخطي على الأسابيع الأقدم فقط. نقارنها بخط أساس ساذج يكرر طلب آخر أسبوع تدريبي. هذا نموذج تنبؤي تمهيدي وليس نظامًا إنتاجيًا أو دليلًا سببيًا.

البيانات الخام تُقرأ للقراءة فقط من `data/dataraw/`. لا يحتاج التحقق إلى `test.csv` لأنه لا يحتوي على الطلب الفعلي.

## 1. قراءة البيانات وتجميع الطلب الأسبوعي

نقرأ `train.csv`، نتحقق من الأعمدة، ثم نجمع `num_orders` على مستوى الأسبوع. لا نحذف أو نرشح سجلات التدريب قبل إنشاء السلسلة الزمنية.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# نحدد مسار البيانات بحسب مجلد تشغيل الدفتر.
project_root = Path.cwd()
if not (project_root / "data" / "dataraw" / "train.csv").exists():
    project_root = project_root.parent
data_path = project_root / "data" / "dataraw" / "train.csv"
assert data_path.exists(), f"ملف التدريب غير موجود: {data_path}"

# نقرأ البيانات الخام للقراءة فقط ونتحقق من الحقول المطلوبة.
train_data = pd.read_csv(data_path)
assert {"week", "num_orders"}.issubset(train_data.columns), "أعمدة week وnum_orders مطلوبة"
print(f"بيانات المصدر: الصفوف = {len(train_data):,} | مجموع num_orders = {train_data['num_orders'].sum():,.0f}")

# نجمع الطلبات الفعلية لكل أسبوع ونرتبها زمنيًا.
weekly_demand = (
    train_data.groupby("week", as_index=False)["num_orders"]
    .sum()
    .sort_values("week")
    .reset_index(drop=True)
)
assert weekly_demand["week"].is_monotonic_increasing, "يجب ترتيب الأسابيع تصاعديًا"
assert weekly_demand["num_orders"].notna().all(), "توجد قيم طلب مفقودة بعد التجميع"
print(f"بعد التجميع الأسبوعي: الصفوف = {len(weekly_demand):,} | مجموع num_orders = {weekly_demand['num_orders'].sum():,.0f}")
print(f"عدد الأسابيع = {len(weekly_demand):,} | النطاق = {weekly_demand['week'].min()}–{weekly_demand['week'].max()}")

## 2. تقسيم زمني للتدريب والتحقق

نحجز آخر 20% من الأسابيع كاملةً للتحقق، ونستخدم كل الأسابيع الأسبق للتدريب. لا نخلط ترتيب الزمن ولا نستخدم أسابيع التحقق لحساب معاملات الاتجاه.

In [ ]:
# نحدد بداية التحقق من أحدث 20 بالمئة من الأسابيع.
validation_weeks = max(1, int(round(len(weekly_demand) * 0.20)))
split_position = len(weekly_demand) - validation_weeks
assert split_position >= 2, "لا تكفي الأسابيع لإنشاء تدريب وتحقق زمني"

# نفصل الأسابيع الأقدم للتدريب والأحدث للتحقق دون خلط زمني.
training_data = weekly_demand.iloc[:split_position].copy()
validation_data = weekly_demand.iloc[split_position:].copy()
print(f"بعد تقسيم التدريب: الصفوف = {len(training_data):,} | مجموع num_orders = {training_data['num_orders'].sum():,.0f}")
print(f"بعد تقسيم التحقق: الصفوف = {len(validation_data):,} | مجموع num_orders = {validation_data['num_orders'].sum():,.0f}")
assert training_data["week"].max() < validation_data["week"].min(), "تسرب زمني بين التدريب والتحقق"
print(f"أسابيع التدريب: {training_data['week'].min()}–{training_data['week'].max()}")
print(f"أسابيع التحقق: {validation_data['week'].min()}–{validation_data['week'].max()}")

## 3. تدريب خط أساس الاتجاه وتقييمه

نقدّر ميلًا واعتراضًا لاتجاه خطي باستخدام بيانات التدريب فقط، ثم نقارن تنبؤه بخط أساس يكرر آخر قيمة تدريبية. نقيس MAE وRMSE وWAPE على أسابيع التحقق؛ هذه المقاييس تخص إجمالي الطلب الأسبوعي فقط.

In [ ]:
# نقدر معاملات الاتجاه الخطي من أسابيع التدريب فقط.
training_week_mean = training_data["week"].mean()
training_demand_mean = training_data["num_orders"].mean()
trend_slope = training_data["week"].cov(training_data["num_orders"]) / training_data["week"].var()
trend_intercept = training_demand_mean - trend_slope * training_week_mean

# ننتج تنبؤ الاتجاه وخط أساس آخر أسبوع على فترة التحقق.
predictions = validation_data[["week", "num_orders"]].rename(columns={"num_orders": "actual_orders"}).copy()
predictions["trend_prediction"] = (
    trend_intercept + trend_slope * predictions["week"]
).clip(lower=0)
predictions["last_week_baseline"] = training_data["num_orders"].iloc[-1]

# نقيس أخطاء التنبؤ مقابل القيم الفعلية المحجوزة.
metric_rows = []
for model_name, prediction_column in [
    ("اتجاه خطي", "trend_prediction"),
    ("تكرار آخر أسبوع تدريبي", "last_week_baseline"),
]:
    errors = predictions[prediction_column] - predictions["actual_orders"]
    metric_rows.append(
        {
            "النموذج": model_name,
            "MAE": errors.abs().mean(),
            "RMSE": errors.pow(2).mean() ** 0.5,
            "WAPE (%)": errors.abs().sum() / predictions["actual_orders"].sum() * 100,
        }
    )
model_metrics = pd.DataFrame(metric_rows).set_index("النموذج")
print(f"معادلة الاتجاه: الطلب المتوقع = {trend_intercept:,.2f} + {trend_slope:,.2f} × week")
print("مقاييس التحقق (وحدة الطلب الأسبوعي):")
print(model_metrics.round(2).to_string())

# نعرض الطلب الفعلي وتنبؤي الاتجاه وآخر أسبوع على المحور الزمني.
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(training_data["week"], training_data["num_orders"], label="التدريب الفعلي", color="#4C72B0")
ax.plot(validation_data["week"], validation_data["num_orders"], label="التحقق الفعلي", color="#55A868")
ax.plot(predictions["week"], predictions["trend_prediction"], label="تنبؤ الاتجاه الخطي", color="#C44E52", linestyle="--")
ax.plot(predictions["week"], predictions["last_week_baseline"], label="خط أساس آخر أسبوع", color="#8172B3", linestyle=":")
ax.axvline(validation_data["week"].min(), color="#777777", linestyle="--", alpha=0.7, label="بداية التحقق")
ax.set_title("الطلب الأسبوعي الفعلي وتوقعات خط الأساس")
ax.set_xlabel("رقم الأسبوع")
ax.set_ylabel("إجمالي num_orders")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
figures_dir = project_root / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
forecast_chart = figures_dir / "weekly_demand_forecast.png"
fig.savefig(forecast_chart, dpi=160, bbox_inches="tight")
print(f"تم حفظ الرسم: {forecast_chart}")
plt.show()
plt.close(fig)

## 4. حدود النموذج

- هذا خط أساس بسيط لإجمالي الطلب على مستوى الأسبوع، وليس نموذجًا معتمدًا أو توقعًا لكل وجبة أو مركز.
- يفترض الاتجاه الخطي استمرار النمط الزمني؛ لا يلتقط الموسمية أو تغيّر الأسعار أو العروض أو اختلاف المراكز.
- قيّمت الدقة على فترة زمنية محجوزة من `train.csv` فقط. لم تُستخدم `test.csv` للتحقق لعدم وجود `num_orders` فيه.
- احسب خطوط أساس موسمية ونماذج إضافية بتقسيمات زمنية متحركة قبل أي استخدام تشغيلي، واربط التقييم بتكلفة النفاد والهدر.
- أرقام `week` تسلسلية وليست تواريخ تقويمية، ولا يجوز تفسيرها كسنة أو موسم فعلي.